<a href="https://colab.research.google.com/github/megamiro-code/battlefield/blob/main/%E5%85%B5%E5%A3%AB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from IPython.display import HTML, display

display(HTML("""
<iframe
    srcdoc='
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<style>
html, body {
    margin: 0;
    width: 100%;
    height: 100%;
    overflow: hidden;
    background: #111;
}

#info {
    position: absolute;
    top: 10px;
    left: 10px;
    z-index: 10;
    color: white;
    background: rgba(0,0,0,0.75);
    padding: 10px 14px;
    font-family: sans-serif;
    border-radius: 6px;
    line-height: 1.5;
}
</style>
</head>

<body>

<div id="info">
10 × 10 Battlefield<br>
🔴 Red Army : 100 Soldiers + Commander<br>
🔵 Blue Army : 100 Soldiers + Commander<br>
<span id="status">Battle ongoing...</span>
</div>

<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js"></script>

<script>

// ============================================================
// 基本設定
// ============================================================

const FIELD_SIZE = 10;
const FIELD_LIMIT = FIELD_SIZE / 2;

const SOLDIER_RADIUS = 0.13;
const COMMANDER_RADIUS = 0.22;

const SOLDIER_HP = 100;
const COMMANDER_HP = 200;

const ATTACK_RANGE = 0.38;
const ATTACK_DAMAGE = 10;

const ATTACK_COOLDOWN = 0.8;

// ランダム移動速度
const SOLDIER_SPEED_MIN = 0.25;
const SOLDIER_SPEED_MAX = 0.45;

const COMMANDER_SPEED = 0.12;

// 攻撃判定時に少し待つ時間
const THINK_INTERVAL_MIN = 0.15;
const THINK_INTERVAL_MAX = 0.35;


// ============================================================
// Scene
// ============================================================

const scene = new THREE.Scene();
scene.background = new THREE.Color(0x20242a);


// ============================================================
// Camera
// ============================================================

const camera = new THREE.PerspectiveCamera(
    50,
    window.innerWidth / window.innerHeight,
    0.1,
    100
);

camera.position.set(0, 15, 5);
camera.lookAt(0, 0, 0);


// ============================================================
// Renderer
// ============================================================

const renderer = new THREE.WebGLRenderer({
    antialias: true
});

renderer.setPixelRatio(window.devicePixelRatio);
renderer.setSize(
    window.innerWidth,
    window.innerHeight
);

document.body.appendChild(renderer.domElement);


// ============================================================
// Light
// ============================================================

scene.add(
    new THREE.AmbientLight(
        0xffffff,
        0.8
    )
);

const light = new THREE.DirectionalLight(
    0xffffff,
    1.0
);

light.position.set(5, 12, 5);
scene.add(light);


// ============================================================
// Ground
// ============================================================

const ground = new THREE.Mesh(
    new THREE.BoxGeometry(
        FIELD_SIZE,
        0.2,
        FIELD_SIZE
    ),
    new THREE.MeshStandardMaterial({
        color: 0xd9d9d9
    })
);

ground.position.y = -0.1;
scene.add(ground);


// ============================================================
// Grid
// ============================================================

for (let i = 0; i <= FIELD_SIZE; i++) {

    const p = -FIELD_LIMIT + i;

    const material =
        new THREE.LineBasicMaterial({
            color: 0x888888
        });

    const geoX =
        new THREE.BufferGeometry();

    geoX.setFromPoints([
        new THREE.Vector3(
            p, 0.01, -FIELD_LIMIT
        ),
        new THREE.Vector3(
            p, 0.01, FIELD_LIMIT
        )
    ]);

    scene.add(
        new THREE.Line(
            geoX,
            material
        )
    );

    const geoZ =
        new THREE.BufferGeometry();

    geoZ.setFromPoints([
        new THREE.Vector3(
            -FIELD_LIMIT, 0.01, p
        ),
        new THREE.Vector3(
            FIELD_LIMIT, 0.01, p
        )
    ]);

    scene.add(
        new THREE.Line(
            geoZ,
            material.clone()
        )
    );
}


// ============================================================
// Corner Walls
// 四隅にだけ壁を配置
// ============================================================

const WALL_SIZE = 1.0;
const WALL_HEIGHT = 0.7;

const wallPositions = [
    [-4.35, -4.35],
    [-4.35,  4.35],
    [ 4.35, -4.35],
    [ 4.35,  4.35]
];

const walls = [];

for (const [x, z] of wallPositions) {

    const wall = new THREE.Mesh(
        new THREE.BoxGeometry(
            WALL_SIZE,
            WALL_HEIGHT,
            WALL_SIZE
        ),
        new THREE.MeshStandardMaterial({
            color: 0x777777
        })
    );

    wall.position.set(
        x,
        WALL_HEIGHT / 2,
        z
    );

    scene.add(wall);

    walls.push({
        x: x,
        z: z,
        halfSize: WALL_SIZE / 2
    });
}


// ============================================================
// Soldier
// ============================================================

function createSoldier(color) {

    const group = new THREE.Group();

    const body = new THREE.Mesh(
        new THREE.BoxGeometry(
            0.22,
            0.28,
            0.22
        ),
        new THREE.MeshStandardMaterial({
            color: color
        })
    );

    body.position.y = 0.15;
    group.add(body);

    const head = new THREE.Mesh(
        new THREE.BoxGeometry(
            0.16,
            0.16,
            0.16
        ),
        new THREE.MeshStandardMaterial({
            color: color
        })
    );

    head.position.y = 0.37;
    group.add(head);

    return group;
}


// ============================================================
// Commander
// ============================================================

function createCommander(color) {

    const group = new THREE.Group();

    const body = new THREE.Mesh(
        new THREE.BoxGeometry(
            0.4,
            0.5,
            0.4
        ),
        new THREE.MeshStandardMaterial({
            color: color
        })
    );

    body.position.y = 0.25;
    group.add(body);

    const head = new THREE.Mesh(
        new THREE.BoxGeometry(
            0.3,
            0.3,
            0.3
        ),
        new THREE.MeshStandardMaterial({
            color: color
        })
    );

    head.position.y = 0.65;
    group.add(head);

    const crown = new THREE.Mesh(
        new THREE.ConeGeometry(
            0.20,
            0.20,
            4
        ),
        new THREE.MeshStandardMaterial({
            color: 0xffd700
        })
    );

    crown.position.y = 0.90;
    group.add(crown);

    return group;
}


// ============================================================
// キャラクター管理
// ============================================================

const units = [];


// ============================================================
// 位置チェック
// ============================================================

function distance2D(x1, z1, x2, z2) {

    const dx = x1 - x2;
    const dz = z1 - z2;

    return Math.sqrt(
        dx * dx + dz * dz
    );
}


// ============================================================
// 壁との衝突判定
// ============================================================

function collidesWithWall(
    x,
    z,
    radius
) {

    for (const wall of walls) {

        const minX =
            wall.x -
            wall.halfSize -
            radius;

        const maxX =
            wall.x +
            wall.halfSize +
            radius;

        const minZ =
            wall.z -
            wall.halfSize -
            radius;

        const maxZ =
            wall.z +
            wall.halfSize +
            radius;

        if (
            x >= minX &&
            x <= maxX &&
            z >= minZ &&
            z <= maxZ
        ) {
            return true;
        }
    }

    return false;
}


// ============================================================
// 他ユニットとの衝突判定
// ============================================================

function collidesWithUnit(
    x,
    z,
    radius,
    ignoreUnit = null
) {

    for (const unit of units) {

        if (
            unit === ignoreUnit ||
            !unit.userData.alive
        ) {
            continue;
        }

        const d =
            distance2D(
                x,
                z,
                unit.position.x,
                unit.position.z
            );

        if (
            d <
            radius +
            unit.userData.radius +
            0.02
        ) {
            return true;
        }
    }

    return false;
}


// ============================================================
// 初期位置探索
// ============================================================

function findFreePosition(
    minX,
    maxX,
    minZ,
    maxZ,
    radius
) {

    for (
        let attempt = 0;
        attempt < 2000;
        attempt++
    ) {

        const x =
            minX +
            Math.random() *
            (maxX - minX);

        const z =
            minZ +
            Math.random() *
            (maxZ - minZ);

        if (
            !collidesWithWall(
                x,
                z,
                radius
            ) &&
            !collidesWithUnit(
                x,
                z,
                radius
            )
        ) {
            return {x, z};
        }
    }

    throw new Error(
        "初期配置可能位置を確保できませんでした"
    );
}


// ============================================================
// Random Direction
// ============================================================

function randomDirection() {

    const angle =
        Math.random() *
        Math.PI * 2;

    return {
        x: Math.cos(angle),
        z: Math.sin(angle)
    };
}


// ============================================================
// Soldier生成
// ============================================================

function createMovingSoldier(
    color,
    team,
    minX,
    maxX,
    minZ,
    maxZ
) {

    const radius =
        SOLDIER_RADIUS;

    const pos =
        findFreePosition(
            minX,
            maxX,
            minZ,
            maxZ,
            radius
        );

    const soldier =
        createSoldier(color);

    soldier.position.set(
        pos.x,
        0,
        pos.z
    );

    const dir =
        randomDirection();

    soldier.userData = {

        team: team,
        type: "soldier",

        hp: SOLDIER_HP,
        maxHp: SOLDIER_HP,

        alive: true,

        radius: radius,

        velocity: {
            x: dir.x,
            z: dir.z
        },

        speed:
            SOLDIER_SPEED_MIN +
            Math.random() *
            (SOLDIER_SPEED_MAX -
             SOLDIER_SPEED_MIN),

        attackCooldown: 0,

        thinkTimer:
            THINK_INTERVAL_MIN +
            Math.random() *
            (THINK_INTERVAL_MAX -
             THINK_INTERVAL_MIN)
    };

    scene.add(soldier);
    units.push(soldier);

    return soldier;
}


// ============================================================
// 赤軍
// ============================================================

for (let i = 0; i < 100; i++) {

    createMovingSoldier(
        0xff3333,
        "red",
        -4.3,
        -0.8,
        -4.3,
        4.3
    );
}


// ============================================================
// 青軍
// ============================================================

for (let i = 0; i < 100; i++) {

    createMovingSoldier(
        0x3388ff,
        "blue",
        0.8,
        4.3,
        -4.3,
        4.3
    );
}


// ============================================================
// Commander
// ============================================================

function createMovingCommander(
    color,
    team,
    x,
    z
) {

    const radius =
        COMMANDER_RADIUS;

    const commander =
        createCommander(color);

    commander.position.set(
        x,
        0,
        z
    );

    const dir =
        randomDirection();

    commander.userData = {

        team: team,
        type: "commander",

        hp: COMMANDER_HP,
        maxHp: COMMANDER_HP,

        alive: true,

        radius: radius,

        velocity: {
            x: dir.x,
            z: dir.z
        },

        speed: COMMANDER_SPEED,

        attackCooldown: 0,

        thinkTimer: 0
    };

    scene.add(commander);
    units.push(commander);

    return commander;
}


const redCommander =
    createMovingCommander(
        0xff0000,
        "red",
        -3.8,
        0
    );


const blueCommander =
    createMovingCommander(
        0x0066ff,
        "blue",
        3.8,
        0
    );


// ============================================================
// 敵ユニット探索
// ============================================================

function findNearestEnemy(unit) {

    let nearest = null;
    let nearestDistance = Infinity;

    for (const other of units) {

        if (
            !other.userData.alive ||
            other.userData.team ===
            unit.userData.team
        ) {
            continue;
        }

        const d =
            distance2D(
                unit.position.x,
                unit.position.z,
                other.position.x,
                other.position.z
            );

        if (d < nearestDistance) {

            nearest = other;
            nearestDistance = d;
        }
    }

    return {
        unit: nearest,
        distance: nearestDistance
    };
}


// ============================================================
// 攻撃
// ============================================================

function attack(attacker, target) {

    if (
        !attacker.userData.alive ||
        !target.userData.alive
    ) {
        return;
    }

    if (
        attacker.userData.attackCooldown > 0
    ) {
        return;
    }

    target.userData.hp -=
        ATTACK_DAMAGE;

    // 攻撃後硬直
    attacker.userData.attackCooldown =
        ATTACK_COOLDOWN;


    // 攻撃時に少しだけ向きを固定
    attacker.userData.thinkTimer =
        0.15;


    if (
        target.userData.hp <= 0
    ) {

        target.userData.hp = 0;
        target.userData.alive = false;

        // 少し沈ませて死亡を表現
        target.position.y = -0.1;

        target.visible = false;


        // 大将死亡
        if (
            target.userData.type ===
            "commander"
        ) {

            endBattle(
                attacker.userData.team
            );
        }
    }
}


// ============================================================
// 勝敗
// ============================================================

let battleEnded = false;

function endBattle(winner) {

    if (battleEnded) {
        return;
    }

    battleEnded = true;

    const status =
        document.getElementById(
            "status"
        );

    status.innerHTML =
        winner === "red"
            ? "<br>🏆 RED WINS!"
            : "<br>🏆 BLUE WINS!";
}


// ============================================================
// 進行方向を決める
// ============================================================

function chooseDirection(unit) {

    const enemy =
        findNearestEnemy(unit);

    if (
        enemy.unit === null
    ) {
        return;
    }

    // 敵が攻撃範囲に入ったら攻撃
    if (
        enemy.distance <=
        ATTACK_RANGE
    ) {

        attack(
            unit,
            enemy.unit
        );

        return;
    }

    // 敵へ向かう
    const dx =
        enemy.unit.position.x -
        unit.position.x;

    const dz =
        enemy.unit.position.z -
        unit.position.z;

    const length =
        Math.sqrt(
            dx * dx +
            dz * dz
        );

    if (length > 0) {

        unit.userData.velocity.x =
            dx / length;

        unit.userData.velocity.z =
            dz / length;
    }
}


// ============================================================
// 1フレーム分の移動を試す
// ============================================================

function tryMove(unit, delta) {

    if (
        !unit.userData.alive
    ) {
        return;
    }

    if (
        unit.userData.attackCooldown >
        0
    ) {
        // 攻撃硬直中は移動不可
        return;
    }

    const v =
        unit.userData.velocity;

    const speed =
        unit.userData.speed;

    const nx =
        unit.position.x +
        v.x * speed * delta;

    const nz =
        unit.position.z +
        v.z * speed * delta;

    const r =
        unit.userData.radius;


    // --------------------------------------------------------
    // フィールド外
    // --------------------------------------------------------

    const limit =
        FIELD_LIMIT - r;

    if (
        nx < -limit ||
        nx > limit ||
        nz < -limit ||
        nz > limit
    ) {

        // 壁際では進行方向を反転
        unit.userData.velocity.x *= -1;
        unit.userData.velocity.z *= -1;

        return;
    }


    // --------------------------------------------------------
    // 壁
    // --------------------------------------------------------

    if (
        collidesWithWall(
            nx,
            nz,
            r
        )
    ) {

        // まずX方向だけ試す
        const onlyX =
            unit.position.x +
            v.x * speed * delta;

        const onlyZ =
            unit.position.z;

        const xOK =
            !collidesWithWall(
                onlyX,
                onlyZ,
                r
            ) &&
            !collidesWithUnit(
                onlyX,
                onlyZ,
                r,
                unit
            );


        // 次にZ方向だけ試す
        const onlyX2 =
            unit.position.x;

        const onlyZ2 =
            unit.position.z +
            v.z * speed * delta;

        const zOK =
            !collidesWithWall(
                onlyX2,
                onlyZ2,
                r
            ) &&
            !collidesWithUnit(
                onlyX2,
                onlyZ2,
                r,
                unit
            );


        if (xOK) {

            unit.position.x =
                onlyX;

            return;
        }

        if (zOK) {

            unit.position.z =
                onlyZ2;

            return;
        }

        return;
    }


    // --------------------------------------------------------
    // 他ユニット
    // --------------------------------------------------------

    if (
        collidesWithUnit(
            nx,
            nz,
            r,
            unit
        )
    ) {
        return;
    }


    // 実際に移動
    unit.position.x = nx;
    unit.position.z = nz;
}


// ============================================================
// Animation
// ============================================================

let previousTime =
    performance.now();

function animate(time) {

    requestAnimationFrame(
        animate
    );

    const delta =
        Math.min(
            (time - previousTime) / 1000,
            0.05
        );

    previousTime = time;


    if (!battleEnded) {

        for (const unit of units) {

            if (
                !unit.userData.alive
            ) {
                continue;
            }


            // クールダウン更新
            unit.userData.attackCooldown =
                Math.max(
                    0,
                    unit.userData.attackCooldown -
                    delta
                );


            // 次の判断まで
            unit.userData.thinkTimer -=
                delta;

            if (
                unit.userData.thinkTimer <= 0
            ) {

                chooseDirection(unit);

                unit.userData.thinkTimer =
                    THINK_INTERVAL_MIN +
                    Math.random() *
                    (THINK_INTERVAL_MAX -
                     THINK_INTERVAL_MIN);
            }


            // 移動
            tryMove(
                unit,
                delta
            );


            // 向いている方向
            const v =
                unit.userData.velocity;

            unit.rotation.y =
                Math.atan2(
                    v.x,
                    v.z
                );
        }
    }


    renderer.render(
        scene,
        camera
    );
}

animate(
    performance.now()
);


// ============================================================
// Resize
// ============================================================

window.addEventListener(
    "resize",
    () => {

        camera.aspect =
            window.innerWidth /
            window.innerHeight;

        camera.updateProjectionMatrix();

        renderer.setSize(
            window.innerWidth,
            window.innerHeight
        );
    }
);

</script>
</body>
</html>'
    style="width:100%; height:650px; border:none;">
</iframe>
"""))